# Delta Live Tables - Introdução

> ⚠️ **Nota:** DLT foi renomeado para **Lakeflow Spark Declarative Pipelines (SDP)**. A certificação pode usar ambos os nomes.

---

## 📊 O que é Delta Live Tables?

**DLT** é um framework **declarativo** para criar pipelines de dados batch e streaming em SQL e Python.

### Abordagem Declarativa vs Imperativa

| Abordagem | Descrição | Exemplo |
|-----------|-----------|---------|
| **Imperativa** | Define COMO fazer (passo a passo) | Spark tradicional |
| **Declarativa** | Define O QUE quer (resultado final) | DLT |

> 💡 **Para prova:** No DLT você declara o resultado desejado e o framework cuida da execução, orquestração e recuperação de falhas.

---

## 🎯 Benefícios do DLT

| Benefício | Descrição |
|-----------|-----------|
| **Orquestração automática** | Gerencia dependências entre tabelas |
| **Compute management** | Escala clusters automaticamente |
| **Monitoramento** | UI integrada com métricas e lineage |
| **Data Quality** | Expectations integradas |
| **Recuperação de falhas** | Retry automático e checkpointing |
| **Batch + Streaming** | Mesmo framework para ambos |

---

## 📋 Requisitos

- Plano **Premium** do Databricks
- Unity Catalog (recomendado)
- Permissão para criar pipelines

---

## 🏗️ Tipos de Datasets no DLT

```
┌─────────────────────────────────────────────────────────────────┐
│                    Delta Live Tables Pipeline                    │
├─────────────────────────────────────────────────────────────────┤
│                                                                  │
│   ┌─────────────────┐   ┌─────────────────┐   ┌──────────────┐ │
│   │ STREAMING TABLE │   │ STREAMING TABLE │   │ MATERIALIZED │ │
│   │    (Bronze)     │──▶│    (Silver)     │──▶│    VIEW      │ │
│   │   Ingestão      │   │  Transformação  │   │   (Gold)     │ │
│   └─────────────────┘   └─────────────────┘   └──────────────┘ │
│                                                                  │
└─────────────────────────────────────────────────────────────────┘
```

### Comparação Rápida

| Tipo | Armazena Dados? | Reprocessa? | Caso de Uso |
|------|-----------------|-------------|-------------|
| **Streaming Table** | ✅ Sim | ❌ Não (append-only) | Ingestão, transformações incrementais |
| **Materialized View** | ✅ Sim | ✅ Sim (recomputa) | Agregações, queries complexas |
| **View** | ❌ Não | N/A (virtual) | Queries intermediárias |

---

## ✅ Expectations (Data Quality)

**Expectations** são regras de qualidade aplicadas a cada registro.

### Três Ações Disponíveis

| Ação SQL | Decorator Python | Comportamento |
|----------|------------------|---------------|
| `EXPECT (condition)` | `@dlt.expect` | Registra violação, **mantém** registro |
| `EXPECT ... ON VIOLATION DROP ROW` | `@dlt.expect_or_drop` | **Remove** registros inválidos |
| `EXPECT ... ON VIOLATION FAIL UPDATE` | `@dlt.expect_or_fail` | **Para** o pipeline |

### Exemplo SQL

```sql
CREATE OR REFRESH STREAMING TABLE silver_orders (
    -- Apenas registra (mantém o registro)
    CONSTRAINT valid_id EXPECT (order_id IS NOT NULL),
    
    -- Remove registros inválidos
    CONSTRAINT valid_amount EXPECT (amount > 0) ON VIOLATION DROP ROW,
    
    -- Para o pipeline se falhar
    CONSTRAINT valid_date EXPECT (order_date IS NOT NULL) ON VIOLATION FAIL UPDATE
)
AS SELECT * FROM STREAM(LIVE.bronze_orders)
```

### Exemplo Python

```python
import dlt

@dlt.table
@dlt.expect("valid_id", "order_id IS NOT NULL")  # Mantém
@dlt.expect_or_drop("valid_amount", "amount > 0")  # Remove
@dlt.expect_or_fail("valid_date", "order_date IS NOT NULL")  # Para
def silver_orders():
    return dlt.read_stream("bronze_orders")
```

### Múltiplas Expectations (Python)

```python
rules = {
    "valid_id": "order_id IS NOT NULL",
    "valid_amount": "amount > 0",
    "valid_date": "order_date IS NOT NULL"
}

@dlt.table
@dlt.expect_all(rules)  # Registra todas
def bronze():
    return ...

@dlt.table
@dlt.expect_all_or_drop(rules)  # Remove se qualquer uma falhar
def silver():
    return ...

@dlt.table
@dlt.expect_all_or_fail(rules)  # Para se qualquer uma falhar
def gold():
    return ...
```

> ⚠️ **Importante para prova:** `EXPECT` sem ação adicional **NÃO remove** dados - apenas registra métricas na UI!

---

## 🔄 Modos de Execução

| Modo | Descrição | Custo | Latência |
|------|-----------|-------|----------|
| **Triggered** | Manual ou agendado | 💰 Menor | ⏱️ Maior |
| **Continuous** | Sempre ativo | 💰💰 Maior | ⏱️ Menor |

---

## ✅ Checklist - Introdução

- [ ] DLT usa abordagem **declarativa**
- [ ] Requer plano **Premium**
- [ ] Suporta **SQL** e **Python**
- [ ] Foi renomeado para **Lakeflow Spark Declarative Pipelines**
- [ ] `EXPECT` sem ação = **mantém** registro (só registra métrica)
- [ ] `ON VIOLATION DROP ROW` = **remove** registro
- [ ] `ON VIOLATION FAIL UPDATE` = **para** pipeline
- [ ] Python múltiplas: `expect_all`, `expect_all_or_drop`, `expect_all_or_fail`

---

## 🔗 Referências

- [O que é Delta Live Tables?](https://learn.microsoft.com/pt-br/azure/databricks/delta-live-tables/)
- [Gerenciar qualidade com Expectations](https://learn.microsoft.com/pt-br/azure/databricks/delta-live-tables/expectations)
- [O que aconteceu com DLT?](https://learn.microsoft.com/pt-br/azure/databricks/ldp/where-is-dlt)